# Session 5 — Numerical Optimization Basics in DICE (Python)

This notebook complements the slides of the **Lecture 2** on carbon pricing and optimal control. We move from scenario-building (*what could happen*) to the social planner's problem (*what should happen*), and show how the **optimal carbon price (SCC)** emerges and is computed numerically.

**Roadmap**  
1. Set up and initialize the DICE model.  
2. Present the model and solve the planner's problem.  
3. Exercise: compare the joint optimum with the no-mitigation optimum.


## Session 5 — Objectives and prerequisites

**Learning objectives**
- Identify the original DICE planner objective, controls, and finite planning window.
- Compare savings-only and joint savings–abatement optimization.
- Read optimal controls and the implied carbon-price path without changing the scientific model.

**Prerequisites.** Slides for Session 5; Session 3 DICE structure; constrained optimization and discounted utility.


## 1. Setup

Run the notebook from this session directory. The original course module and data files are stored next to it.

**Convention.** At the initial date, control variables are `NaN`: the initial state is given, but the control is chosen only for subsequent transitions.


In [ ]:
# Install dependencies in the current notebook kernel before running imports.
%pip install -r requirements.txt


### 1.A Setup
We import the DICE model components and ensure Python can find the module.


In [ ]:

import os, sys, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make sure we can import the user's module
from importlib import reload
import DICE
reload(DICE)

from DICE import Params, init_states, update_path, obj_fun, run_optimal_policy, mat_to_df
print("DICE loaded from:", DICE.__file__)


## 2. Presentation

### 2.A Model skeleton (states & controls)

- **States (examples):** capital $K_t$, TFP $A_t$, population $L_t$, climate stocks $M_{\mathrm{AT}}, M_{\mathrm{UP}}, M_{\mathrm{LO}}$, temperatures $T_{\mathrm{AT}}, T_{\mathrm{LO}}$.
- **Controls:** saving rate $s_t$, abatement rate $\mu_t$ (which, together with technology parameters, implies a **carbon price / tax** path).
- **Units:** $C_t$ in trillion USD, $L_t$ in million → per-capita consumption in **thousand USD**: $c_t = 1000 \times C_t/L_t$.


In [ ]:

# Instantiate parameters and construct an initial path
p = Params()
sim = init_states(p)

# Baseline controls for initialization (not optimal): small abatement, constant saving
sim[1:, p.i_mu] = 0.03
sim[1:, p.i_s]  = 0.20

# Update path for all endogenous variables given the controls
timevec = range(1, p.nT)
sim = update_path(sim, timevec, p)

# Peek at available columns
print("Columns:", p.col)
print("Horizon length (periods):", p.nT, " — years:", sim[-1, p.i_time])


### 2.B Planner's objective (CRRA utility)

We maximize discounted social welfare
$$
W = \sum_{i=0}^{I^\star} \beta_\Delta^{\,i}\, L_{t+i\Delta}\, u\!\left( 1000\,\frac{C_{t+i\Delta}}{L_{t+i\Delta}} \right), 
\qquad \beta_\Delta=(1+\rho)^{-\Delta},
$$
with CRRA utility $u(c)=\frac{c^{1-\gamma}-1}{1-\gamma}$. The notebook uses the `obj_fun` provided in the module.

**Note on truncation:** We replace the infinite horizon with a finite $I^\star$ chosen so the tail weight is below a tolerance.


In [ ]:
# Evaluate discounted welfare with a constant saving rate of 20%.
# obj_fun returns NEGATIVE welfare because the optimizer minimizes it.
x_const = np.full(p.nT - 1, 0.20)
W_val = -obj_fun(x_const, sim, timevec, p, [p.i_s])
start_year = int(sim[timevec[0], p.i_time])
end_year = int(sim[timevec[-1], p.i_time])
print(f"Discounted welfare evaluated at {start_year}, over {start_year}–{end_year}, is {W_val:.2f} (constant saving rate: 20%).")
# Total discounted welfare over the horizon, not utility in one period.


### 2.C Finite-horizon planning window $I^{\star}$

We select a planning window so that the discount weight of the last term is below the numerical tolerance `p.toly`. This implements the finite-horizon approximation discussed in the slides.


In [ ]:

# Compute planning window similar to DICE.run_optimal_policy
disc, Tplanner = 1.0, 1
while disc > p.toly:
    Tplanner += 1
    disc *= (1.0/(1.0 + p.rho))**p.Delta

print("Chosen planning window (periods):", Tplanner, " (each period = Δ years =", p.Delta, ")")


### 2.D Solve the full problem: **savings + abatement**

Now we optimize the pair $(s_t, \mu_t)$. The carbon tax path is then implied by the model and reported in the `Tax` column. The problem reads as:

$$
\begin{aligned}
\max_{\{s_{t+i},\,\{\mu_{t+i}\}_{i=0}^{I^\star}} \quad 
& \sum_{i=0}^{I^\star} \beta^{i}\, L_{t+i\Delta}\;
u\!\left( 1000\,\frac{C_{t+i\Delta}}{L_{t+i\Delta}} \right) \\[0.4em]
\text{s.t.}\quad 
& y_{t+i} \;=\; G_p\!\big(y_{t+i-1},\, x_{t+i-1},\, s_{t+i},\, \mu_{t+i};\big), 
\quad i=0,\dots,I^\star\!-\!1,\\
& s_{\min} \;\le\; s_{t+i} \;\le\; s_{\max}\ \ (\text{box constraints}),\\
& 0 \;\le\; \mu_{t+i} \;\le\; 1\ \ (\text{box constraints}),\\
& y_{t-1} \;=\; y^{\mathrm{given}}_{t-1},
\end{aligned}
$$


In [ ]:
# 2.D — Joint optimum: the planner chooses both saving and abatement.
sim_smu = sim.copy()
# bounds_smu   = [(p.s_lower, p.s_upper), (0.0, 1.0)]
# control_id   = [p.i_s, p.i_mu]
# path_opt_smu = run_optimal_policy(sim_smu, timevec, p, bounds_smu, control_id)


### 2.E Visualize the optimal controls and the implied carbon price

Below we plot (one chart per figure) the optimal saving \(s_t\), abatement \(\mu_t\), and the implied carbon tax (USD/tCO\(_2\)) reported in `Tax`.


In [ ]:
# Plot the joint optimum after you have completed Exercise 2.D.
# Until then, this cell deliberately remains executable without revealing the solution.
if "path_opt_smu" not in globals():
    print("Complete Exercise 2.D and run its cell before producing these plots.")
else:
    # Plot saving rate
    plt.figure(figsize=(6,3.8))
    plt.plot(path_opt_smu[:, p.i_time], path_opt_smu[:, p.i_s], label="Optimal transition (s_t)")
    plt.xlabel("Year"); plt.ylabel("Saving rate s_t"); plt.title("Optimal saving rate")
    plt.legend(); plt.grid(True); plt.tight_layout()

    # Plot abatement rate
    plt.figure(figsize=(6,3.8))
    plt.plot(path_opt_smu[:, p.i_time], path_opt_smu[:, p.i_mu], label="Optimal transition (μ_t)")
    plt.xlabel("Year"); plt.ylabel("Abatement rate μ_t"); plt.title("Optimal abatement rate")
    plt.legend(); plt.grid(True); plt.tight_layout()

    # Plot implied carbon tax (USD per tCO2)
    plt.figure(figsize=(6,3.8))
    plt.plot(path_opt_smu[:, p.i_time], path_opt_smu[:, p.i_Tax], label="Implied carbon tax (USD/tCO₂)")
    plt.xlabel("Year"); plt.ylabel("Carbon tax (USD/tCO₂)"); plt.title("Optimal carbon price / SCC (model-implied)")
    plt.legend(); plt.grid(True); plt.tight_layout()


## 3. Exercise — Optimal climate policy versus no-mitigation optimum

### 3.A Joint optimal policy

This exercise compares the **joint optimum**, in which the planner chooses saving and abatement, with a **constrained optimum without mitigation**, in which the planner chooses saving but abatement is fixed at zero.

> This second regime is not strict laissez-faire: saving remains a planner's choice. It isolates the value of having the mitigation instrument available.

Reuse the joint optimum computed above, in which the planner chooses both $s_t$ and $\mu_t$.


In [ ]:
# 3.A — Reuse the joint optimum computed in Section 6.
# path_opt_full = ...


### 3.B Optimal saving with abatement fixed at zero

Construct the constrained optimum with no mitigation:

1. start from a copy of the initial simulation;
2. impose $\mu_t=0$ for every transition period;
3. remove p.i_mu from control_id, so that the optimizer chooses only saving;
4. solve the problem and store the path as path_opt_no_mitigation.


In [ ]:
# 3.B — Constrained optimum without mitigation.
# sim_no_mitigation = sim.copy()
# sim_no_mitigation[1:, p.i_mu] = ...
# path_opt_no_mitigation = run_optimal_policy(
#     sim_no_mitigation, timevec, p, (p.s_lower, p.s_upper), [p.i_s]
# )


### 3.C Compare the two policy regimes

Produce a three-panel figure comparing the joint optimum and the no-mitigation optimum for atmospheric temperature $T_{AT,t}$, industrial emissions $E_t$, and consumption $C_t$. Briefly explain why the two regimes differ.


In [ ]:
# 3.C — Plot T_AT, E, and C for the two policy regimes.
# Hint: use plt.subplots(1, 3, figsize=(15, 4)) and the columns
# p.i_T_AT, p.i_E, and p.i_C.


### 3.D Timing of mitigation costs and climate benefits

Compute the consumption gap and temperature gap:

$$100\left(\frac{C_t^{\mathrm{joint}}}{C_t^{\mathrm{no\ mitigation}}}-1\right),
\qquad T_t^{\mathrm{joint}}-T_t^{\mathrm{no\ mitigation}}.$$

Plot both gaps and comment: when does the consumption cost appear, and when does the climate benefit become visible?


In [ ]:
# 3.D — Compute and plot the consumption and temperature gaps.
# consumption_gap = 100 * (
#     path_opt_full[:, p.i_C] / path_opt_no_mitigation[:, p.i_C] - 1
# )
# temperature_gap = ...
